# VITASA_Enhanced — Training on Google Colab
**Đề tài:** Enhancing Vietnamese Targeted Aspect Sentiment Analysis with Social Media Text Normalization and Imbalanced Learning

**Lịch chạy (mỗi session ~2h free GPU):**
| Ngày | Domain | Configs |
|------|--------|---------|
| Ngày 1 | `mobile` | C1, C2, C3, C4 |
| Ngày 2 | `restaurant` | C1, C2, C3, C4 |
| Ngày 3 | `hotel` | C1, C2, C3, C4 |

**Thứ tự chạy mỗi ngày:** Cell 1 → 2 → 3 → 4 (test) → 5 (train domain đó) → 6 (xem kết quả) → 7 (download)

In [ ]:
# ── Cell 1: Kiểm tra GPU ──────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('VRAM       :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  Không có GPU — vào Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone repo + install dependencies ─────────────────────────────────
# Nếu VM reset (mất /content/VITASA_Enhanced) thì chạy lại cell này
import os
if not os.path.exists('/content/VITASA_Enhanced'):
    !git clone https://github.com/Hunganh1305/VITASA_Enhanced.git /content/VITASA_Enhanced
else:
    !git -C /content/VITASA_Enhanced pull
%cd /content/VITASA_Enhanced
!pip install -r requirements.txt -q
print('\n✅ Setup done')

In [ ]:
# ── Cell 3: Verify project ────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['find', '.', '-type', 'f', '-name', '*.py', '-o', '-name', '*.jsonl'],
                        capture_output=True, text=True, cwd='/content/VITASA_Enhanced')
print('Project files:')
for f in sorted(result.stdout.strip().split('\n')):
    print(' ', f)

print('\nRunning tests...')
!python -m pytest text_normalization/tests/ imbalanced_learning/tests/ -q 2>&1 | tail -3

In [ ]:
# ── Cell 4: Test nhanh 1 config (2 epoch) — xác nhận pipeline OK ─────────────
# Chạy cell này trước, nếu dev F1 > 0% thì pipeline OK
!python train.py --domain mobile --loss ce --epochs 2
print('\n✅ Test run OK — sẵn sàng chạy full')

In [ ]:
# ── Cell 5: Training — thay DOMAIN mỗi ngày ───────────────────────────────────
# ⭐ CHỈ CẦN ĐỔI DÒNG NÀY MỖI NGÀY:
#   Ngày 1 → DOMAIN = 'mobile'
#   Ngày 2 → DOMAIN = 'restaurant'
#   Ngày 3 → DOMAIN = 'hotel'
DOMAIN = 'mobile'

# ─────────────────────────────────────────────────────────────────────────────
import subprocess
import time

configs = [
    ('ce',    False, 'C1 — Baseline (CE)'),
    ('ce',    True,  'C2 — + Text Norm'),
    ('focal', False, 'C3 — + Focal Loss'),
    ('focal', True,  'C4 — + Both (Full Model)'),
]

EPOCHS = 10
overall_start = time.time()

print(f'🚀 Training domain: {DOMAIN.upper()} — {len(configs)} configs × {EPOCHS} epochs')
print('=' * 70)

for i, (loss, normalize, label) in enumerate(configs, 1):
    norm_flag = '--normalize' if normalize else ''
    cmd = f'python train.py --domain {DOMAIN} --loss {loss} {norm_flag} --epochs {EPOCHS}'
    cmd_parts = [p for p in cmd.split() if p]  # remove empty strings from split

    print(f'\n[{i}/4] {label}')
    print(f'CMD: {cmd}')
    print('-' * 70)

    t0 = time.time()
    result = subprocess.run(cmd_parts, cwd='/content/VITASA_Enhanced')
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f'❌ Config [{label}] FAILED (exit code {result.returncode})')
        break
    print(f'✅ Done in {elapsed/60:.1f} min')

total = time.time() - overall_start
print(f'\n{"=" * 70}')
print(f'✅ {DOMAIN.upper()} done in {total/60:.1f} min ({total/3600:.1f} hours)')
print(f'{"=" * 70}')

In [ ]:
# ── Cell 6: Tổng hợp kết quả (chạy sau mỗi ngày hoặc khi xong hết) ───────────
import json
from pathlib import Path

results_dir = Path('/content/VITASA_Enhanced/experiments/results')
rows = []

for result_file in sorted(results_dir.glob('*/results.json')):
    with open(result_file) as f:
        d = json.load(f)
    rows.append({
        'domain'  : d['domain'],
        'config'  : ('C4' if d['normalize'] and d['loss'] == 'focal' else
                     'C3' if not d['normalize'] and d['loss'] == 'focal' else
                     'C2' if d['normalize'] else 'C1'),
        'normalize': '✅' if d['normalize'] else '❌',
        'loss'    : d['loss'],
        'dev_f1'  : round(d['best_dev_f1'] * 100, 2),
        'test_f1' : round(d['test_f1'] * 100, 2),
    })

if not rows:
    print('⚠️  Chưa có kết quả nào — chạy Cell 5 trước')
else:
    print(f'{"-" * 72}')
    print(f'{"Domain":<12} {"Config":<6} {"Norm":<6} {"Loss":<12} {"Dev F1":>8} {"Test F1":>8}')
    print(f'{"-" * 72}')
    for r in sorted(rows, key=lambda x: (x['domain'], x['config'])):
        print(f'{r["domain"]:<12} {r["config"]:<6} {r["normalize"]:<6} {r["loss"]:<12} {r["dev_f1"]:>7.2f}% {r["test_f1"]:>7.2f}%')
    print(f'{"-" * 72}')
    print(f'Total: {len(rows)}/12 configs done')

In [ ]:
# ── Cell 7: Download kết quả (chạy sau mỗi ngày để lưu về máy) ───────────────
# Chỉ zip results.json (bỏ qua best_model.pt ~400MB)
!find /content/VITASA_Enhanced/experiments/results -name 'results.json' | zip /content/results_summary.zip -@

from google.colab import files
files.download('/content/results_summary.zip')
print('✅ Download started')